# PS07-AI — Bài 15: từ mô hình hộp đến CLT
**Giảng viên: Nguyễn Bích Vân — IAI-UET-VNU**

 Dữ liệu giả lập, không cần CSV đầu vào. Chạy từ trên xuống; ví dụ học cú pháp dùng bộ số riêng, không phải đáp án của bài tập.

## Chuẩn bị
Mở cả thư mục đã giải nén trong VS Code, chọn kernel Python rồi chạy từng ô. Kết quả hiển thị ngay dưới ô mã. Nếu thiếu thư viện, chạy `%pip install -r requirements.txt` trong một ô mới rồi khởi động lại kernel.

Không sao chép dấu nhắc `PS ...>` hoặc `>>` vào ô Python. Các ô `None` là vị trí cần tự hoàn thiện; khi còn thiếu, phần tương ứng chỉ nhắc và không xuất CSV rỗng.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import norm
from plotnine import (ggplot, aes, after_stat, geom_density,
                      geom_histogram, geom_line, labs)
from IPython.display import display
print("Đã nạp thư viện.")

Đã nạp thư viện.


## 0. Ví dụ học cú pháp — số liệu khác đề
Mỗi hàng dưới đây là một lô gồm ba lượt. `axis=1` đi dọc các cột của từng hàng; `axis=0` tổng hợp các hàng tại cùng vị trí cột. `ddof=0` mô tả phân phối của toàn bộ các thẻ đồng khả năng trong hộp đã biết.

In [2]:
demo_box = np.array([1, 1, 3, 5])
demo_rng = np.random.default_rng(15)
demo_samples = demo_rng.choice(demo_box, size=(8, 3), replace=True)
print("Mảng có dạng:", demo_samples.shape)
print(demo_samples)
print("Tổng từng lô:", demo_samples.sum(axis=1))
print("Trung bình từng lô:", demo_samples.mean(axis=1))
print("Trung bình hộp:", demo_box.mean())
print("Phương sai hộp:", demo_box.var(ddof=0))

Mảng có dạng: (8, 3)
[[5 3 3]
 [5 1 1]
 [1 1 1]
 [3 5 1]
 [5 3 1]
 [1 3 1]
 [1 5 1]
 [5 1 5]]
Tổng từng lô: [11  7  3  9  9  5  7 11]
Trung bình từng lô: [3.66666667 2.33333333 1.         3.         3.         1.66666667
 2.33333333 3.66666667]
Trung bình hộp: 2.5
Phương sai hộp: 2.75


## 1. Hoàn thiện cấu hình Bài 15a–b
Nhập đúng 12 thẻ từ Bài 6, giữ cả những giá trị lặp. Tự chọn `axis` và `ddof`, sau đó giải thích bằng lời ở cuối notebook. Các cỡ mẫu và số lô đã đặt theo đề.

In [3]:
CONFIG_BOX = {
    "box": None,               # Điền danh sách 12 thẻ từ Bài 6.
    "axis": None,              # Tổng hợp theo chiều nào của mảng (B, n)?
    "ddof": None,              # Mẫu số dùng cho phương sai của chiếc hộp.
    "sample_sizes": [16, 64],
    "repetitions": 5000,
    "seed": 2026,
}
box_ready = all(CONFIG_BOX[k] is not None for k in ["box", "axis", "ddof"])
if not box_ready:
    print("Cần điền box, axis và ddof trong CONFIG_BOX.")

Cần điền box, axis và ddof trong CONFIG_BOX.


In [4]:
box_summary = None
box_means = None
rng = np.random.default_rng(CONFIG_BOX["seed"])
if box_ready:
    box = np.asarray(CONFIG_BOX["box"], dtype=float)
    if box.ndim != 1 or len(box) != 12 or not np.isfinite(box).all():
        raise ValueError("Hộp phải là một danh sách 12 giá trị hữu hạn.")
    if CONFIG_BOX["axis"] not in [0, 1] or CONFIG_BOX["ddof"] not in [0, 1]:
        raise ValueError("axis và ddof phải là 0 hoặc 1; hãy giải thích lựa chọn.")
    B = CONFIG_BOX["repetitions"]
    mu = box.mean()
    sigma = box.std(ddof=CONFIG_BOX["ddof"])
    records, frames = [], []
    for n in CONFIG_BOX["sample_sizes"]:
        samples = rng.choice(box, size=(B, n), replace=True)
        totals = samples.sum(axis=CONFIG_BOX["axis"])
        means = samples.mean(axis=CONFIG_BOX["axis"])
        if totals.shape != (B,) or means.shape != (B,):
            raise ValueError("Kết quả chưa có B lô. Xem lại ý nghĩa của axis.")
        records.append({
            "sample_size": n,
            "total_mean_empirical": totals.mean(), "total_mean_theory": n*mu,
            "total_sd_empirical": totals.std(ddof=0),
            "total_sd_theory": sigma*np.sqrt(n),
            "mean_mean_empirical": means.mean(), "mean_mean_theory": mu,
            "mean_sd_empirical": means.std(ddof=0),
            "mean_sd_theory": sigma/np.sqrt(n),
        })
        frames.append(pd.DataFrame({"sample_mean": means, "sample_size": str(n)}))
    box_summary = pd.DataFrame(records)
    box_means = pd.concat(frames, ignore_index=True)
    output_dir = Path.cwd() / "student_outputs"
    output_dir.mkdir(exist_ok=True)
    box_summary.to_csv(output_dir / "box_summary.csv", index=False)
    box_means.to_csv(output_dir / "box_sample_means.csv", index=False)
    display(box_summary.round(6))
else:
    print("Chưa mô phỏng: hoàn thiện ô CONFIG_BOX trước.")

Chưa mô phỏng: hoàn thiện ô CONFIG_BOX trước.


## 2. Hai mật độ của trung bình — Bài 15c
`color="sample_size"` trong `aes` phân nhóm theo cỡ mẫu. Mỗi nhóm được ước lượng một mật độ riêng; không phải số đếm thô của nhóm. So sánh cả tâm, độ rộng và hình dạng.

In [5]:
if box_means is not None:
    p_means = (ggplot(box_means, aes(x="sample_mean", color="sample_size"))
               + geom_density()
               + labs(x="Trung bình số lượt gọi công cụ", y="Mật độ", color="Cỡ mẫu n"))
    p_means.show()
    p_means.save(output_dir / "box_mean_densities.png", width=8, height=4.8,
                 dpi=160, verbose=False)
else:
    print("Chưa có dữ liệu để vẽ; không tạo tệp rỗng.")

Chưa có dữ liệu để vẽ; không tạo tệp rỗng.


## 3. Chuẩn hóa trung bình các biến mũ — Bài 15d
Dùng kỳ vọng và SD cá thể đã tìm ở Bài 12–13 để điền cấu hình. `scale` của NumPy là thời gian trung bình; độ lệch chuẩn dùng để chuẩn hóa trung bình phải có cỡ mẫu $n$ trong công thức.

In [6]:
CONFIG_CLT = {
    "mean_wait": None,         # Kỳ vọng cá thể theo đề (giây).
    "sd_wait": None,           # SD cá thể, không phải phương sai.
    "sample_size": 30,
    "repetitions": 5000,
}
clt_ready = all(CONFIG_CLT[k] is not None for k in ["mean_wait", "sd_wait"])
z_data = None
if clt_ready and box_summary is not None:
    mean_wait, sd_wait = CONFIG_CLT["mean_wait"], CONFIG_CLT["sd_wait"]
    if mean_wait <= 0 or sd_wait <= 0:
        raise ValueError("Kỳ vọng và SD phải dương.")
    n = CONFIG_CLT["sample_size"]
    samples = rng.exponential(scale=mean_wait, size=(CONFIG_CLT["repetitions"], n))
    means = samples.mean(axis=1)
    z = (means - mean_wait) / (sd_wait / np.sqrt(n))
    z_data = pd.DataFrame({"z": z})
    z_data.to_csv(output_dir / "clt_standardized_means.csv", index=False)
    print("Trung bình Z:", z.mean(), "| SD của Z:", z.std(ddof=0))
else:
    print("Điền mean_wait, sd_wait và hoàn thành phần hộp trước khi chạy CLT.")

Điền mean_wait, sd_wait và hoàn thành phần hộp trước khi chạy CLT.


In [7]:
if z_data is not None:
    grid = np.linspace(min(-4.0, z.min()), max(4.0, z.max()), 500)
    reference = pd.DataFrame({"z": grid, "density": norm.pdf(grid)})
    p_clt = (ggplot(z_data, aes(x="z", y=after_stat("density")))
             + geom_histogram(bins=40)
             + geom_line(data=reference, mapping=aes(x="z", y="density"),
                         inherit_aes=False)
             + labs(x="Trung bình đã chuẩn hóa Z", y="Mật độ"))
    p_clt.show()
    p_clt.save(output_dir / "clt_standardized_histogram.png", width=8, height=4.8,
               dpi=160, verbose=False)
else:
    print("Chưa có Z để vẽ.")

Chưa có Z để vẽ.


## 4. Bài viết của sinh viên
**15a.** Giải thích `ddof`, PMF và vì sao phải giữ thẻ lặp: _viết ở đây_.

**15b–c.** So sánh bảng số và hai mật độ: _viết ở đây_.

**15d.** Giải thích `scale`, `axis=1` và công thức chuẩn hóa: _viết ở đây_.

**15e.** Viết 5–7 câu nhận xét theo đề; nêu ít nhất một lỗi mô hình mà phần mềm vẫn chạy được: _viết ở đây_.

Chạy lại từ đầu trước khi nộp. Nộp cả notebook, bảng CSV và hai hình PNG trong `student_outputs/`.